# 01 — Data Foundation
**Levante Ferries · portfolio ficticio · 2026**

Ejecutar desde la raíz del proyecto o desde `notebooks`. Cada tabla se calcula sobre CSV regenerables.

In [1]:
from pathlib import Path
import sys
ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
sys.path.insert(0,str(ROOT/'src'))
import pandas as pd
import numpy as np
from model import assess,aggregate,calculate,optimize,FUELEU_LIMIT
DATA=ROOT/'data'; OUT=ROOT/'outputs'
voyages=pd.read_csv(DATA/'voyages_2026_synthetic.csv')
print('Raíz:',ROOT.name,'| viajes:',len(voyages),'| periodo:',voyages.date.min(),voyages.date.max())

Raíz: levante_ferries_sustainability | viajes: 4860 | periodo: 2026-01-01 2026-08-31


## Propósito
Un viaje programado es la unidad de decisión. Datos 100 % ficticios y reproducibles con semilla 20260923. Cruce explícito de ruta, buque, combustible, demanda, coste y alcance regulatorio; los IDs de otros proyectos no se cruzan.

In [2]:
routes=pd.read_csv(DATA/'routes.csv'); vessels=pd.read_csv(DATA/'vessels.csv'); fuels=pd.read_csv(DATA/'fuel_assumptions.csv')
print('Viajes, rutas, buques y combustibles:',len(voyages),len(routes),len(vessels),len(fuels))
print(voyages.groupby('route').size().to_string())

Viajes, rutas, buques y combustibles: 4860 8 10 4
route
Algeciras – Tánger    486
Barcelona – Ibiza     486
Barcelona – Palma     486
Dénia – Formentera    486
Dénia – Ibiza         972
Dénia – Mallorca      486
Valencia – Orán       486
Valencia – Palma      972


In [3]:
assert voyages.voyage_id.is_unique
assert voyages.isna().sum().sum()==0
assert (voyages.passengers<=voyages.pax_capacity).all() and (voyages.cargo_lm<=voyages.cargo_capacity_lm).all()
assert (voyages.distance_nm>0).all() and set(voyages.route_id)==set(routes.route_id)
assert (voyages.vessel_id.map(vessels.set_index('vessel_id').route_id)==voyages.route_id).all()
print('Validación: unicidad, completitud, capacidad y relaciones coherentes')

Validación: unicidad, completitud, capacidad y relaciones coherentes


**Límite:** la demanda, las tarifas y los costes son simulados. Forecasting, pricing y finanzas previos aportan contratos conceptuales, no datos enlazados fila a fila.